#### 0. Setup de variables

In [1]:
import sys

sys.path.append("..")
from utils import SimpleLogger

logger = SimpleLogger(log_name="web_scraping")

In [2]:
from utils import Constantes

# Links para el Web Scraping
lk_wr = Constantes.lk_wr
lk_cs_scores = Constantes.lk_cs_scrs
lk_en_scores = Constantes.lk_en_scrs
lk_cs_stats = Constantes.lk_cs_stts
lk_en_stats = Constantes.lk_en_stts

# Columnas a extraer
scores_titles = Constantes.scores_titles
scores_ws_wr = Constantes.scores_ws_wr
scores_ws_su = Constantes.scores_ws_su
stats_titles = Constantes.stats_titles
stats_ws_wr = Constantes.stats_ws_wr
stats_ws_su = Constantes.stats_ws_su

print(f"lk_wr: {lk_wr}")
print(f"lk_cs_scores: {lk_cs_scores}")
print(f"lk_en_scores: {lk_en_scores}")
print(f"lk_cs_stats: {lk_cs_stats}")
print(f"lk_en_stats: {lk_en_stats}")

print(f"scores_titles: {scores_titles}")
print(f"scores_ws_wr: {scores_ws_wr}")
print(f"scores_ws_su: {scores_ws_su}")
print(f"stats_titles: {stats_titles}")
print(f"stats_ws_wr: {stats_ws_wr}")
print(f"stats_ws_su: {stats_ws_su}")

lk_wr: https://www.timeshighereducation.com/world-university-rankings/2025/world-ranking
lk_cs_scores: https://www.timeshighereducation.com/world-university-rankings/2025/subject-ranking/computer-science#!/length/-1/sort_by/rank/sort_order/asc/cols/scores
lk_en_scores: https://www.timeshighereducation.com/world-university-rankings/2025/subject-ranking/engineering-and-it#!/length/-1/sort_by/rank/sort_order/asc/cols/scores
lk_cs_stats: https://www.timeshighereducation.com/world-university-rankings/2025/subject-ranking/computer-science#!/length/-1/sort_by/rank/sort_order/asc/cols/stats
lk_en_stats: https://www.timeshighereducation.com/world-university-rankings/2025/subject-ranking/engineering-and-it#!/length/-1/sort_by/rank/sort_order/asc/cols/stats
scores_titles: ['rank', 'name', 'country', 'overall_score', 'research_qlt_score', 'industry_score', 'intl_outlook_score', 'research_env_score', 'teaching_score']
scores_ws_wr: ['cell-rank css-.*', 'cell-name css-.*', 'cell-scores_overall css-.

#### 2. Postprocesamiento

##### Funciones para postprocesamiento

In [3]:
import pandas as pd
import numpy as np


def max_overall_score(score_str: str) -> str:
    if isinstance(score_str, str):
        # Use '–' (en dash) as observed in the image, not '-'
        if "–" in score_str:
            parts = score_str.split("–")
            return float(parts[1])
        # Fallback for hyphen if en dash not present
        elif "-" in score_str:
            parts = score_str.split("-")
            return float(parts[1])
        else:
            try:
                return float(score_str)
            except ValueError:
                return None
    # Return as is if not a string (e.g., already a float or NaN)
    return score_str


def convertir_ranking(df: pd.DataFrame, in_col: str) -> pd.DataFrame:
    # Ubicar los casos de rankings con intervalos (e.g. 201-250)
    is_range = df[in_col].str.contains("–", regex=False)
    # Ubicar los casos de rankings que terminan con + (e.g. 1001+)
    is_plus = df[in_col].str.contains("\\+", regex=True)

    # Etapa 1: el = que antecede al número (e.g. =14 -> 14)
    if not df.loc[~is_range, in_col].empty:
        df.loc[~is_range, in_col] = df.loc[~is_range, in_col].str.replace("=", "")

    # Etapa 2: intervalos a autoincrementales (e.g. 201-250 -> 201)
    df_aux = df.copy()
    if not df_aux.loc[is_range, in_col].empty:
        # Separar en dos columnas 201 y 250 (lower y upper, respectivamente)
        # La tercera columna es un autoincremental
        bounds = df_aux.loc[is_range, in_col].astype(str).str.split("–", expand=True)
        df_aux["lower"] = bounds[0].astype(int)
        df_aux["upper"] = bounds[1].astype(int)
        df_aux["temp_block_id"] = (df_aux["rank"] != df_aux["rank"].shift()).cumsum()

        for block_id, group in df_aux.groupby("temp_block_id"):
            lower_bound = group["lower"].iloc[0]
            upper_bound = group["upper"].iloc[0]
            sequence_start = lower_bound
            sequence_length = len(group)
            if str(sequence_start) != "nan":
                generated_sequence = np.arange(
                    sequence_start, sequence_start + sequence_length
                )
                final_sequence = np.minimum(generated_sequence, upper_bound)
                df.loc[group.index, in_col] = final_sequence

    # Etapa 3: el + después de números y autoincremental (e.g. 1001+ -> 1001, 1002, ...)
    df_aux = df.copy()
    if not df_aux.loc[is_plus, in_col].empty:
        df_aux["start_rank"] = (
            df_aux.loc[is_plus, in_col].astype(str).str.replace("\\+", "", regex=True)
        )
        df_aux["temp_block_id"] = (
            df_aux.loc[is_plus, in_col] != df_aux.loc[is_plus, in_col].shift()
        ).cumsum()

        for block_id, group in df_aux.groupby("temp_block_id"):
            start_rank = int(group["start_rank"].iloc[0])
            sequence_length = len(group)
            if str(start_rank) != "nan":
                generated_sequence = np.arange(start_rank, start_rank + sequence_length)
                df.loc[group.index, in_col] = generated_sequence.astype(int)
    return df

##### Parte principal

In [4]:
import pandas as pd

from utils import Constantes

ruta_csv = f"{Constantes.ruta_reports}/raw"

df_wr_scores = pd.read_csv(f"{ruta_csv}/wr_scores.csv")
df_wr_stats = pd.read_csv(f"{ruta_csv}/wr_stats.csv")
df_cs_scores = pd.read_csv(f"{ruta_csv}/cs_scores.csv")
df_cs_stats = pd.read_csv(f"{ruta_csv}/cs_stats.csv")
df_en_scores = pd.read_csv(f"{ruta_csv}/en_scores.csv")
df_en_stats = pd.read_csv(f"{ruta_csv}/en_stats.csv")

##### wr

In [5]:
# Convertir de object a str los valores de rank, name y country
in_col_1 = "rank"
in_col_2 = "name"
in_col_3 = "country"
df_wr_scores[in_col_1] = df_wr_scores[in_col_1].astype(str)
df_wr_scores[in_col_2] = df_wr_scores[in_col_2].astype(str)
df_wr_scores[in_col_3] = df_wr_scores[in_col_3].astype(str)
# Convertir str =10 a int 10, 201-250 a 201 y autoincrementales, y 1001+
# a 1001 y autoincrementales
in_col_4 = in_col_1
df_wr_scores = convertir_ranking(df_wr_scores, in_col_4)
df_wr_scores[in_col_4] = df_wr_scores[in_col_4].astype(int)
# Convertir de str a float el valor de overall_score
in_col_5 = "overall_score"
df_wr_scores[in_col_5] = df_wr_scores[in_col_5].apply(max_overall_score).astype(float)
# Resultados
print(df_wr_scores.shape)
df_wr_scores.head(1)

(966, 9)


,rank,name,country,overall_score,research_qlt_score,industry_score,intl_outlook_score,research_env_score,teaching_score
0,1,University of Oxford,United Kingdom,98.5,98.8,99.6,97.3,100.0,96.8


In [6]:
# Convertir de object a str los valores de rank, name y country
in_col_1 = "rank"
in_col_2 = "name"
in_col_3 = "country"
df_wr_stats[in_col_1] = df_wr_stats[in_col_1].astype(str)
df_wr_stats[in_col_2] = df_wr_stats[in_col_2].astype(str)
df_wr_stats[in_col_3] = df_wr_stats[in_col_3].astype(str)
# Convertir str =10 a int 10, 201-250 a 201 y autoincrementales, y 1001+
# a 1001 y autoincrementales
in_col_4 = in_col_1
df_wr_stats = convertir_ranking(df_wr_stats, in_col_4)
df_wr_stats[in_col_4] = df_wr_stats[in_col_4].astype(int)
# Resultados
print(df_wr_stats.shape)
df_wr_stats.head(1)

(966, 7)


,rank,name,country,number_students,student_staff_ratio,intl_students,female_male_ratio
0,1,University of Oxford,United Kingdom,"22,095",10.8,43%,51 : 49


In [7]:
df_wr = pd.merge(
    df_wr_scores,
    df_wr_stats,
    on=["rank", "name", "country"],
    how="outer",
    indicator=True,
)
print(df_wr.shape)
df_wr.head(1)

(966, 14)


,rank,name,country,overall_score,research_qlt_score,industry_score,intl_outlook_score,research_env_score,teaching_score,number_students,student_staff_ratio,intl_students,female_male_ratio,_merge
0,1,University of Oxford,United Kingdom,98.5,98.8,99.6,97.3,100.0,96.8,"22,095",10.8,43%,51 : 49,both


In [8]:
missing_df_wr_scores = df_wr[df_wr["_merge"] == "right_only"].drop(columns=["_merge"])
missing_df_wr_stats = df_wr[df_wr["_merge"] == "left_only"].drop(columns=["_merge"])
print(f"Faltantes en scores: {missing_df_wr_scores.shape}")
print(f"Faltantes en stats: {missing_df_wr_stats.shape}")

Faltantes en scores: (0, 13)
Faltantes en stats: (0, 13)


##### cs

In [9]:
# Convertir de object a str los valores de rank, name y country
in_col_1 = "rank"
in_col_2 = "name"
in_col_3 = "country"
df_cs_scores[in_col_1] = df_cs_scores[in_col_1].astype(str)
df_cs_scores[in_col_2] = df_cs_scores[in_col_2].astype(str)
df_cs_scores[in_col_3] = df_cs_scores[in_col_3].astype(str)
# Convertir str =10 a int 10, 201-250 a 201 y autoincrementales, y 1001+
# a 1001 y autoincrementales
in_col_4 = in_col_1
df_cs_scores = convertir_ranking(df_cs_scores, in_col_4)
df_cs_scores[in_col_4] = df_cs_scores[in_col_4].astype(int)
# Convertir de str a float el valor de overall_score
in_col_5 = "overall_score"
df_cs_scores[in_col_5] = df_cs_scores[in_col_5].apply(max_overall_score).astype(float)
# Resultados
print(df_cs_scores.shape)
df_cs_scores.head(1)

(1122, 9)


,rank,name,country,overall_score,research_qlt_score,industry_score,intl_outlook_score,research_env_score,teaching_score
0,1,University of Oxford,United Kingdom,98.3,99.4,92.4,95.5,98.7,99.2


In [10]:
# Convertir de object a str los valores de rank, name y country
in_col_1 = "rank"
in_col_2 = "name"
in_col_3 = "country"
df_cs_stats[in_col_1] = df_cs_stats[in_col_1].astype(str)
df_cs_stats[in_col_2] = df_cs_stats[in_col_2].astype(str)
df_cs_stats[in_col_3] = df_cs_stats[in_col_3].astype(str)
# Convertir str =10 a int 10, 201-250 a 201 y autoincrementales, y 1001+
# a 1001 y autoincrementales
in_col_4 = in_col_1
df_cs_stats = convertir_ranking(df_cs_stats, in_col_4)
df_cs_stats[in_col_4] = df_cs_stats[in_col_4].astype(int)
# Resultados
print(df_cs_stats.shape)
df_cs_stats.head(1)

(1122, 7)


,rank,name,country,number_students,student_staff_ratio,intl_students,female_male_ratio
0,1,University of Oxford,United Kingdom,"22,095",10.8,43%,51 : 49


In [11]:
df_cs = pd.merge(
    df_cs_scores,
    df_cs_stats,
    on=["rank", "name", "country"],
    how="outer",
    indicator=True,
)
print(df_cs.shape)
df_cs.head(1)

(1122, 14)


,rank,name,country,overall_score,research_qlt_score,industry_score,intl_outlook_score,research_env_score,teaching_score,number_students,student_staff_ratio,intl_students,female_male_ratio,_merge
0,1,University of Oxford,United Kingdom,98.3,99.4,92.4,95.5,98.7,99.2,"22,095",10.8,43%,51 : 49,both


In [12]:
missing_df_cs_scores = df_cs[df_cs["_merge"] == "right_only"].drop(columns=["_merge"])
missing_df_cs_stats = df_cs[df_cs["_merge"] == "left_only"].drop(columns=["_merge"])
print(f"Faltantes en scores: {missing_df_cs_scores.shape}")
print(f"Faltantes en stats: {missing_df_cs_stats.shape}")

Faltantes en scores: (0, 13)
Faltantes en stats: (0, 13)


##### en

In [13]:
# Convertir de object a str los valores de rank, name y country
in_col_1 = "rank"
in_col_2 = "name"
in_col_3 = "country"
df_en_scores[in_col_1] = df_en_scores[in_col_1].astype(str)
df_en_scores[in_col_2] = df_en_scores[in_col_2].astype(str)
df_en_scores[in_col_3] = df_en_scores[in_col_3].astype(str)
# Convertir str =10 a int 10, 201-250 a 201 y autoincrementales, y 1001+
# a 1001 y autoincrementales
in_col_4 = in_col_1
df_en_scores = convertir_ranking(df_en_scores, in_col_4)
df_en_scores[in_col_4] = df_en_scores[in_col_4].astype(int)
# Convertir de str a float el valor de overall_score
in_col_5 = "overall_score"
df_en_scores[in_col_5] = df_en_scores[in_col_5].apply(max_overall_score).astype(float)
# Resultados
print(df_en_scores.shape)
df_en_scores.head(1)

(1488, 9)


,rank,name,country,overall_score,research_qlt_score,industry_score,intl_outlook_score,research_env_score,teaching_score
0,1,Harvard University,United States,97.5,97.6,97.1,93.7,99.6,96.5


In [14]:
# Convertir de object a str los valores de rank, name y country
in_col_1 = "rank"
in_col_2 = "name"
in_col_3 = "country"
df_en_stats[in_col_1] = df_en_stats[in_col_1].astype(str)
df_en_stats[in_col_2] = df_en_stats[in_col_2].astype(str)
df_en_stats[in_col_3] = df_en_stats[in_col_3].astype(str)
# Convertir str =10 a int 10, 201-250 a 201 y autoincrementales, y 1001+
# a 1001 y autoincrementales
in_col_4 = in_col_1
df_en_stats = convertir_ranking(df_en_stats, in_col_4)
df_en_stats[in_col_4] = df_en_stats[in_col_4].astype(int)
# Resultados
print(df_en_stats.shape)
df_en_stats.head(1)

(1488, 7)


,rank,name,country,number_students,student_staff_ratio,intl_students,female_male_ratio
0,1,Harvard University,United States,"22,584",10.0,25%,52 : 48


In [15]:
df_en = pd.merge(
    df_en_scores,
    df_en_stats,
    on=["rank", "name", "country"],
    how="outer",
    indicator=True,
)
print(df_en.shape)
df_en.head(1)

(1488, 14)


,rank,name,country,overall_score,research_qlt_score,industry_score,intl_outlook_score,research_env_score,teaching_score,number_students,student_staff_ratio,intl_students,female_male_ratio,_merge
0,1,Harvard University,United States,97.5,97.6,97.1,93.7,99.6,96.5,"22,584",10.0,25%,52 : 48,both


In [16]:
missing_df_en_scores = df_en[df_en["_merge"] == "right_only"].drop(columns=["_merge"])
missing_df_en_stats = df_en[df_en["_merge"] == "left_only"].drop(columns=["_merge"])
print(f"Faltantes en scores: {missing_df_en_scores.shape}")
print(f"Faltantes en stats: {missing_df_en_stats.shape}")

Faltantes en scores: (0, 13)
Faltantes en stats: (0, 13)


##### Guardar rankings en csv

In [17]:
from utils import Constantes
import os

ruta_csv = f"{Constantes.ruta_reports}/staging"
os.makedirs(ruta_csv, exist_ok=True)

df_wr.to_csv(f"{ruta_csv}/wr.csv", index=False)
df_cs.to_csv(f"{ruta_csv}/cs.csv", index=False)
df_en.to_csv(f"{ruta_csv}/en.csv", index=False)